# DistilBERT Training Notebook
Notebook do treningu modelu przewidującego ceny samochodów.

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datasets import Dataset

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
df = pd.read_csv('../data/train.csv')
df.head()

In [ ]:
def row_to_text(row):
    return (
        f"Year: {row['year']}, "
        f"Brand: {row['brand']}, "
        f"Mileage: {row['mileage']} km, "
        f"Fuel: {row['fuel']}, "
        f"Province: {row['province']}"
    )

df['text'] = df.apply(row_to_text, axis=1)
df['price'] = np.log1p(df['price'])
df[['text', 'price']].head()

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[['text', 'price']])
test_dataset = Dataset.from_pandas(test_df[['text', 'price']])

train_dataset = train_dataset.rename_column('price', 'labels')
test_dataset = test_dataset.rename_column('price', 'labels')

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type='regression'
)

In [ ]:
training_args = TrainingArguments(
    output_dir='../outputs/model',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    weight_decay=0.01
)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    rmse = mean_squared_error(labels, predictions, squared=False)
    mae = mean_absolute_error(labels, predictions)
    r2 = r2_score(labels, predictions)

    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()